# Investigating Low Wine Quality with PySpark

This demo uses the open UCI Red Wine Quality dataset to explore a practical question:

> Which measured characteristics help predict whether a wine receives a low quality score, and what patterns should we investigate further?

We will:

1. Load the data and define **low quality** as a score of 5 or below.
2. Train a PySpark random forest.
3. Evaluate it on unseen data using AUC.
4. Rank measurements using the model's built-in feature importance.
5. Examine the observed low-quality rates for alcohol, volatile acidity, and sulphates.

**Important:** Feature importance identifies useful *predictors*, not proven causes of wine quality.

In [1]:
from urllib.request import urlretrieve

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler

spark = SparkSession.builder.appName("WineQualityFeatureImportance").getOrCreate()

## 1. Load the red wine dataset

The UCI CSV uses a **semicolon (`;`)** as its separator. Each row represents a wine sample. The `quality` column is the recorded quality score and will be used to create our prediction target.

In [2]:
url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "wine-quality/winequality-red.csv"
)

urlretrieve(url, "winequality-red.csv")

wine_df = spark.read.csv(
    "winequality-red.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

wine_df.printSchema()
wine_df.show(5)
print("Number of wine samples:", wine_df.count())

root
 |-- fixed acidity: double (nullable = true)
 |-- volatile acidity: double (nullable = true)
 |-- citric acid: double (nullable = true)
 |-- residual sugar: double (nullable = true)
 |-- chlorides: double (nullable = true)
 |-- free sulfur dioxide: double (nullable = true)
 |-- total sulfur dioxide: double (nullable = true)
 |-- density: double (nullable = true)
 |-- pH: double (nullable = true)
 |-- sulphates: double (nullable = true)
 |-- alcohol: double (nullable = true)
 |-- quality: integer (nullable = true)

+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|fixed acidity|volatile acidity|citric acid|residual sugar|chlorides|free sulfur dioxide|total sulfur dioxide|density|  pH|sulphates|alcohol|quality|
+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|          7.4|          

## 2. Define what we mean by “low quality”

We create a binary label:

- `1.0`: quality score **5 or below**
- `0.0`: quality score **above 5**

The original `quality` column must **not** be included as a model feature: it directly determines the label and would leak the answer to the model.

In [3]:
wine_df = wine_df.withColumn(
    "low_quality",
    (F.col("quality") <= 5).cast("double")
)

feature_columns = [
    column
    for column in wine_df.columns
    if column not in ("quality", "low_quality")
]

print("Features used by the model:")
print(feature_columns)

wine_df.groupBy("low_quality").count().orderBy("low_quality").show()

Features used by the model:
['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']
+-----------+-----+
|low_quality|count|
+-----------+-----+
|        0.0|  855|
|        1.0|  744|
+-----------+-----+



## 3. Train a random forest

The pipeline has two stages:

1. `VectorAssembler` combines the measurements into one feature vector.
2. `RandomForestClassifier` predicts whether a wine is low quality.

We train on approximately **80%** of the samples and reserve **20%** for evaluation. This lets us check whether the model works on wines it did not see during training.

In [4]:
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

forest = RandomForestClassifier(
    featuresCol="features",
    labelCol="low_quality",
    numTrees=100,
    seed=42
)

pipeline = Pipeline(stages=[assembler, forest])

train_df, test_df = wine_df.randomSplit([0.8, 0.2], seed=42)

model = pipeline.fit(train_df)
predictions = model.transform(test_df)

print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

predictions.select(
    "quality", "low_quality", "probability", "prediction"
).show(10, truncate=False)

Training rows: 1324
Test rows: 275
+-------+-----------+----------------------------------------+----------+
|quality|low_quality|probability                             |prediction|
+-------+-----------+----------------------------------------+----------+
|7      |0.0        |[0.8706096712648725,0.12939032873512743]|0.0       |
|6      |0.0        |[0.8115720820820231,0.1884279179179769] |0.0       |
|5      |1.0        |[0.63588482338546,0.36411517661454]     |0.0       |
|5      |1.0        |[0.3469043148698531,0.6530956851301468] |1.0       |
|7      |0.0        |[0.8766468175138288,0.1233531824861712] |0.0       |
|7      |0.0        |[0.8234289288044175,0.17657107119558246]|0.0       |
|5      |1.0        |[0.3009423911593146,0.6990576088406855] |1.0       |
|5      |1.0        |[0.6339272436711088,0.36607275632889125]|0.0       |
|6      |0.0        |[0.7612771374284336,0.23872286257156633]|0.0       |
|6      |0.0        |[0.7132181099780786,0.28678189002192145]|0.0       |
+--

## 4. Check predictive performance

AUC measures how well the model ranks low-quality wines above higher-quality wines across different decision thresholds.

AUC is **not** the same as accuracy. We evaluate using `rawPrediction`, which contains the model's prediction scores rather than only its final 0-or-1 decisions.

In [5]:
evaluator = BinaryClassificationEvaluator(
    labelCol="low_quality",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(predictions)
print(f"Test AUC: {auc:.3f}")

# AUC = 0.858. That suggests the model distinguishes the two groups reasonably well on the held-out sample.

Test AUC: 0.858


## 5. Find measurements the model relied on

PySpark's fitted random forest provides `featureImportances`. A higher score means the feature contributed more to the forest's splits.

These scores do **not** tell us:

- whether *higher* or *lower* values are associated with low quality;
- whether changing a measurement would improve quality;
- the causal impact of a measurement.

To understand direction, we will inspect the data after ranking the features.

In [6]:
importances = model.stages[-1].featureImportances.toArray()

ranking = sorted(
    zip(feature_columns, importances),
    key=lambda item: item[1],
    reverse=True
)

print("Feature importance ranking:")
for feature, importance in ranking:
    print(f"{feature:25s} {importance:.4f}")

Feature importance ranking:
alcohol                   0.3816
sulphates                 0.1731
volatile acidity          0.1244
total sulfur dioxide      0.0845
density                   0.0618
fixed acidity             0.0381
chlorides                 0.0328
pH                        0.0311
citric acid               0.0309
free sulfur dioxide       0.0235
residual sugar            0.0182


For example, 0.3816 means alcohol received approximately 38% of this forest’s total importance score. It does not mean alcohol causes 38% of wine quality.


## 6. Investigate alcohol

Feature importance tells us alcohol matters to the model, but not **which direction** the relationship goes.

We group wines in the test set by alcohol level and calculate the percentage labeled low quality in each group.

In [7]:
alcohol_summary = (
    test_df
    .withColumn(
        "alcohol_group",
        F.when(F.col("alcohol") < 10, "Below 10")
         .when(F.col("alcohol") < 11, "10 to under 11")
         .otherwise("11 or above")
    )
    .groupBy("alcohol_group")
    .agg(
        F.count("*").alias("wine_count"),
        F.round(
            F.avg("low_quality") * 100, 1
        ).alias("low_quality_percent")
    )
    .orderBy(F.desc("low_quality_percent"))
)

alcohol_summary.show()

+--------------+----------+-------------------+
| alcohol_group|wine_count|low_quality_percent|
+--------------+----------+-------------------+
|      Below 10|       116|               71.6|
|10 to under 11|        85|               42.4|
|   11 or above|        74|               14.9|
+--------------+----------+-------------------+



Results showed that 71.6% of wines below 10 alcohol were labeled low quality, compared with 14.9% of wines at 11 or above.

## 7. Investigate volatile acidity and sulphates

For these measurements, we use the **training data** to determine cutoffs at approximately the one-third and two-thirds points. We then apply those cutoffs to the **test data**.

This avoids choosing boundaries after looking at the test-set results. The groups are used to describe observed patterns, not to prove cause and effect.

In [8]:
for feature in ["volatile acidity", "sulphates"]:
    lower, upper = train_df.approxQuantile(
        feature,
        [1 / 3, 2 / 3],
        0.01
    )

    print(f"\n{feature}")
    print(f"Low < {lower:.3f}; High >= {upper:.3f}")

    summary = (
        test_df
        .withColumn(
            "measurement_group",
            F.when(F.col(feature) < lower, "1 Low")
             .when(F.col(feature) < upper, "2 Medium")
             .otherwise("3 High")
        )
        .groupBy("measurement_group")
        .agg(
            F.count("*").alias("wine_count"),
            F.round(
                F.avg("low_quality") * 100, 1
            ).alias("low_quality_percent")
        )
        .orderBy("measurement_group")
    )

    summary.show()


volatile acidity
Low < 0.430; High >= 0.600
+-----------------+----------+-------------------+
|measurement_group|wine_count|low_quality_percent|
+-----------------+----------+-------------------+
|            1 Low|        88|               23.9|
|         2 Medium|        94|               51.1|
|           3 High|        93|               65.6|
+-----------------+----------+-------------------+


sulphates
Low < 0.580; High >= 0.670
+-----------------+----------+-------------------+
|measurement_group|wine_count|low_quality_percent|
+-----------------+----------+-------------------+
|            1 Low|       101|               69.3|
|         2 Medium|        56|               46.4|
|           3 High|       118|               28.8|
+-----------------+----------+-------------------+



## 8. Interpretation

In this run:

| Measurement | Observed relationship with low-quality rate |
|---|---|
| Alcohol | Lower alcohol groups had higher low-quality rates. |
| Volatile acidity | Higher volatile-acidity groups had higher low-quality rates. |
| Sulphates | Lower-sulphate groups had higher low-quality rates. |

The workflow is:

**Train a predictive model → identify important measurements → inspect the direction of observed patterns → decide what warrants further investigation.**

### Limitations

- **Association is not causation.** Raising alcohol or sulphates, or lowering volatile acidity, would not necessarily improve a wine's quality.
- Measurements may be related to one another, so an individual importance score should not be treated as an independent effect.
- Results may change with a different train/test split or random-forest configuration.
- The grouped results describe this dataset and its recorded quality scores; they are not a production recommendation by themselves.